# GDELT API Explorer

Reference: see `API_USAGE.md` in this folder.

In [1]:
from urllib.parse import urlencode

SOURCE_NAME = "GDELT"
BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"
CAPABILITIES = [
    "Global news monitoring across many languages and regions.",
    "Querying by keywords, entities, themes, and source country.",
    "Support for article lists, timelines, and tone analysis outputs.",
    "High-frequency coverage suitable for event-driven studies.",
]
DEFAULT_QUERY_PARAMS = {
    "query": '("Ukraine" OR "Russia") sourcelang:English',
    "mode": "ArtList",
    "format": "json",
    "maxrecords": "10",
    "sort": "DateDesc",
}

print(SOURCE_NAME)
print("Preview only: this cell does not fetch API records.")
print("Capabilities:")
for item in CAPABILITIES:
    print("-", item)
print("\nDefault query params:")
for key, value in DEFAULT_QUERY_PARAMS.items():
    print(f"- {key}: {value}")
print("\nQuery URL preview:")
print(f"{BASE_URL}?{urlencode(DEFAULT_QUERY_PARAMS)}")

GDELT
Preview only: this cell does not fetch API records.
Capabilities:
- Global news monitoring across many languages and regions.
- Querying by keywords, entities, themes, and source country.
- Support for article lists, timelines, and tone analysis outputs.
- High-frequency coverage suitable for event-driven studies.

Default query params:
- query: ("Ukraine" OR "Russia") sourcelang:English
- mode: ArtList
- format: json
- maxrecords: 10
- sort: DateDesc

Query URL preview:
https://api.gdeltproject.org/api/v2/doc/doc?query=%28%22Ukraine%22+OR+%22Russia%22%29+sourcelang%3AEnglish&mode=ArtList&format=json&maxrecords=10&sort=DateDesc


In [2]:
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import time

LIVE_QUERY_PARAMS = {
    "query": "economy",
    "mode": "ArtList",
    "format": "json",
    "maxrecords": "5",
    "sort": "DateDesc",
}

request_url = f"{BASE_URL}?{urlencode(LIVE_QUERY_PARAMS)}"
print("Live request URL:")
print(request_url)

payload = None
last_error = None
for attempt_index, delay_seconds in enumerate([0, 5, 10, 20], start=1):
    if delay_seconds > 0:
        print(f"Waiting {delay_seconds}s before retry...")
        time.sleep(delay_seconds)
    try:
        request = Request(request_url, headers={"User-Agent": "news-api-explorer/1.0"})
        with urlopen(request, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))
        print(f"Fetch succeeded on attempt {attempt_index}.")
        break
    except HTTPError as error:
        last_error = error
        print(f"Attempt {attempt_index} failed with HTTP {error.code}.")
        if error.code != 429:
            break
    except Exception as error:
        last_error = error
        print(f"Attempt {attempt_index} failed: {error}")
        break

if payload is None:
    print("No live payload returned.")
    print(f"Last error: {last_error}")
else:
    articles = payload.get("articles", [])
    print(f"\nArticles returned: {len(articles)}")
    for index, article in enumerate(articles, start=1):
        title = article.get("title", "<missing title>")
        url = article.get("url", "<missing url>")
        print(f"\n{index}. {title}")
        print(url)

    output_dir = Path("outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "gdelt_live_response.json"
    output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"\nSaved payload to: {output_path.resolve()}")

Live request URL:
https://api.gdeltproject.org/api/v2/doc/doc?query=economy&mode=ArtList&format=json&maxrecords=5&sort=DateDesc
Fetch succeeded on attempt 1.

Articles returned: 5

1. IHSG Kamis Pagi Rebound , Ikuti Penguatan Bursa Asia dan Global
https://ekonomi.republika.co.id/berita/tbeos1370/ihsg-kamis-pagi-rebound-ikuti-penguatan-bursa-asia-dan-global

2. Dubai nỗ lực duy trì nhịp sống du lịch giữa vùng xung đột
https://baomoi.com/dubai-no-luc-duy-tri-nhip-song-du-lich-giua-vung-xung-dot-c54612764.epi

3. Ash - Sharq : база США вблизи иракского Эрбиля подверглась атаке с воздуха
https://news.mail.ru/society/70041139/

4.  【 理响中国 】 凝心聚力谱写中国式现代化新篇章 热烈祝贺十四届全国人大四次会议开幕 - 南海网
https://www.hinews.cn/page?n=2804373&m=1&s=1044

5. 中国今年军费预算增长7 %， 达19095 . 61亿元
https://sputniknews.cn/20260305/1070071523.html

Saved payload to: /Users/gwh/projects/news/notebooks/api_explorer/gdelt/outputs/gdelt_live_response.json
